In [26]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "/Users/jaswanth/KKbox/dataset"
OUTPUT_DIR = "/Users/jaswanth/KKbox/dataset/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option("display.max_columns", 50)


In [27]:
train = pd.read_csv(os.path.join(DATA_DIR, "train_v2.csv"))
members = pd.read_csv(os.path.join(DATA_DIR, "members_v3.csv"))

print("train_v2.csv:", train.shape)
print("members_v3.csv:", members.shape)


train_v2.csv: (970960, 2)
members_v3.csv: (6769473, 6)


In [28]:
train.head()


,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [29]:
members.head()


,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


In [30]:
def inspect(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nNull counts:")
    print(df.isnull().sum())
    print("\n")

inspect(train, "train_v2")
inspect(members, "members_v3")


--- train_v2 ---
Shape: (970960, 2)

Dtypes:
msno          str
is_churn    int64
dtype: object

Null counts:
msno        0
is_churn    0
dtype: int64


--- members_v3 ---
Shape: (6769473, 6)

Dtypes:
msno                        str
city                      int64
bd                        int64
gender                      str
registered_via            int64
registration_init_time    int64
dtype: object

Null counts:
msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64




In [31]:
# Churn label balance — important since churn is typically imbalanced
train["is_churn"].value_counts(normalize=True)


is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64

**Note:** if churn is a small minority class (commonly ~5-10% for this dataset), remember this
downstream when choosing evaluation metrics — accuracy alone will be misleading.


In [32]:
SAMPLE_SIZE = 25000   # adjust up/down depending on your machine's memory
RANDOM_STATE = 42

# Stratified sample so churn/non-churn ratio is preserved
# (using groupby().sample() directly, not groupby().apply(), since newer pandas
# versions can drop the grouping column when using .apply())
sample_frac = SAMPLE_SIZE / len(train)
train_sample = (
    train.groupby("is_churn", group_keys=False)
    .sample(frac=sample_frac, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Sampled users:", train_sample.shape)
print(train_sample["is_churn"].value_counts(normalize=True))


Sampled users: (25000, 2)
is_churn
0    0.91004
1    0.08996
Name: proportion, dtype: float64


In [33]:
member_ids = set(train_sample["msno"])
len(member_ids)


25000

In [34]:
members_sample = members[members["msno"].isin(member_ids)].copy()
print("members_sample:", members_sample.shape)
members_sample.head()


members_sample: (22085, 6)


,msno,city,bd,gender,registered_via,registration_init_time
137,eIOUZ5I+NV/3EDfn/U/tMepn4FJt2SdzOrWGH1tNlYI=,5,24,male,3,20141025
1189,RnFRxb3wVe+2w87QRsaKDNPMfNXEfspldnbjkVepu7M=,1,0,NaN,7,20170126
1322,nq+4KRKNWTQkH9VNArdNfhBNl70Vh01WEi/i9rPlxqU=,1,0,NaN,13,20170201
1630,iQyU+ZPerEmWCx0uv46m8C+vPKBiK9esX6sTTUMGfbg=,13,23,male,9,20070506
2320,oYYSuEi8Ib0MP2/Dn/alYNTUyXsQvtpd91uxJewhtGU=,22,23,male,9,20070806


In [35]:
CHUNK_SIZE = 500_000

transaction_chunks = []
for chunk in pd.read_csv(os.path.join(DATA_DIR, "transactions_v2.csv"), chunksize=CHUNK_SIZE):
    filtered = chunk[chunk["msno"].isin(member_ids)]
    if len(filtered) > 0:
        transaction_chunks.append(filtered)

transactions_sample = pd.concat(transaction_chunks, ignore_index=True)
print("transactions_sample:", transactions_sample.shape)
transactions_sample.head()


transactions_sample: (29336, 9)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
1,+3Z/2l0S7ui1s9FlZgsPYPUm3VOBhZGgW5toALP8VOg=,41,30,149,149,1,20150731,20171202,0
2,+SYHxcHwzn28S6jWZv8IKGG0p6/hEf0RzywL8K5rqbU=,41,30,149,149,1,20170306,20170406,0
3,+TTn0VJn1O/b9MPpiRgL5sHvNzDVsn7vvLN/A89dRiY=,39,30,149,149,1,20170228,20170419,0
4,/BndJt9YSOh1kzEykXyHrQZKl943rqFrzR9efW2b7wE=,39,30,149,149,1,20170228,20170424,0


In [36]:
log_chunks = []
for i, chunk in enumerate(pd.read_csv(os.path.join(DATA_DIR, "user_logs_v2.csv"), chunksize=CHUNK_SIZE)):
    filtered = chunk[chunk["msno"].isin(member_ids)]
    if len(filtered) > 0:
        log_chunks.append(filtered)
    if i % 10 == 0:
        print(f"Processed chunk {i}, matches so far: {sum(len(c) for c in log_chunks)}")

user_logs_sample = pd.concat(log_chunks, ignore_index=True)
print("user_logs_sample:", user_logs_sample.shape)
user_logs_sample.head()


Processed chunk 0, matches so far: 9412
Processed chunk 10, matches so far: 104325
Processed chunk 20, matches so far: 198529
Processed chunk 30, matches so far: 292973
user_logs_sample: (347294, 9)


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,bLf43Q9GcbNT+8UVbhTZ41lXUnoXsHQDIVSuWVviYhg=,20170329,0,0,1,0,33,30,8457.177
1,bB/I5ukaNnz6ZCxMQj1Zg2EUF9jfJyaRKECS/bTfssA=,20170304,173,24,7,1,2,186,5506.589
2,GHkCF5oViCOdi3Jh/Lwx+Wwz9YzIcyNVPrAeZGvZs1U=,20170305,125,41,13,4,5,155,9188.734
3,TQrCNstL0vMrhLuRY3+iBZKK4dB+CgRci9Unu1+1ztc=,20170315,1,0,2,4,142,47,35825.873
4,VyAIzSeB4Ixl9lAjnxHnlNapycqOwXX7hIXyvJ/ds7A=,20170309,25,26,15,12,122,143,38761.128


In [37]:
print("Before cleaning:")
print(members_sample["bd"].describe())

VALID_AGE_MIN, VALID_AGE_MAX = 10, 90
members_sample["bd_clean"] = members_sample["bd"].where(
    members_sample["bd"].between(VALID_AGE_MIN, VALID_AGE_MAX), np.nan
)

print("\nAfter cleaning:")
print(members_sample["bd_clean"].describe())
print("Missing after cleaning:", members_sample["bd_clean"].isnull().mean().round(3), "fraction")


Before cleaning:
count    22085.000000
mean        13.620693
std         17.590949
min        -48.000000
25%          0.000000
50%          0.000000
75%         27.000000
max       1032.000000
Name: bd, dtype: float64

After cleaning:
count    9983.000000
mean       29.894120
std         8.929032
min        13.000000
25%        24.000000
50%        28.000000
75%        34.000000
max        90.000000
Name: bd_clean, dtype: float64
Missing after cleaning: 0.548 fraction


In [38]:
members_sample["registration_init_time"] = pd.to_datetime(
    members_sample["registration_init_time"], format="%Y%m%d", errors="coerce"
)

transactions_sample["transaction_date"] = pd.to_datetime(
    transactions_sample["transaction_date"], format="%Y%m%d", errors="coerce"
)
transactions_sample["membership_expire_date"] = pd.to_datetime(
    transactions_sample["membership_expire_date"], format="%Y%m%d", errors="coerce"
)

user_logs_sample["date"] = pd.to_datetime(
    user_logs_sample["date"], format="%Y%m%d", errors="coerce"
)


In [39]:
transactions_agg = (
    transactions_sample.sort_values("transaction_date")
    .groupby("msno")
    .agg(
        num_transactions=("transaction_date", "count"),
        last_transaction_date=("transaction_date", "max"),
        last_expire_date=("membership_expire_date", "max"),
        last_plan_price=("plan_list_price", "last"),
        last_amount_paid=("actual_amount_paid", "last"),
        last_payment_method=("payment_method_id", "last"),
        last_payment_plan_days=("payment_plan_days", "last"),
        is_auto_renew=("is_auto_renew", "last"),
        any_cancel=("is_cancel", "max"),
    )
    .reset_index()
)

print("transactions_agg:", transactions_agg.shape)
transactions_agg.head()


transactions_agg: (24042, 10)


,msno,num_transactions,last_transaction_date,last_expire_date,last_plan_price,last_amount_paid,last_payment_method,last_payment_plan_days,is_auto_renew,any_cancel
0,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2,2017-03-31,2017-05-19,149,149,39,30,1,0
1,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,1,2017-03-26,2017-04-26,149,149,41,30,1,0
2,++boFsOAGvAI3O+P4RG9O+p7e/dF7JdGb6/b+mf0ahk=,2,2017-03-31,2017-04-13,129,129,30,30,1,1
3,++l8WoNUmsqs7C9ZVyk3pdxXklhdJcSrj50v5su2x/w=,1,2017-03-31,2017-04-30,99,99,41,30,1,0
4,++xwmVgCwpW+wr4+zw30CGbaTGYdt7yD9y+TFNI+NVs=,1,2017-03-23,2017-04-22,149,149,40,30,1,0


### 7.4 Aggregate user logs per user

Same idea — reduce daily logs to one summary row per user (raw aggregates here; deeper
behavioral features like listening trend belong in the feature-engineering notebook).


In [40]:
logs_agg = (
    user_logs_sample.groupby("msno")
    .agg(
        active_days=("date", "nunique"),
        first_log_date=("date", "min"),
        last_log_date=("date", "max"),
        total_secs_sum=("total_secs", "sum"),
        total_secs_mean=("total_secs", "mean"),
        num_unq_mean=("num_unq", "mean"),
        num_100_sum=("num_100", "sum"),
        num_25_sum=("num_25", "sum"),
    )
    .reset_index()
)

print("logs_agg:", logs_agg.shape)
logs_agg.head()


logs_agg: (19362, 9)


,msno,active_days,first_log_date,last_log_date,total_secs_sum,total_secs_mean,num_unq_mean,num_100_sum,num_25_sum
0,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,28,2017-03-01,2017-03-31,115411.260,4121.830714,16.714286,485,43
1,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,21,2017-03-02,2017-03-30,149896.558,7137.931333,39.428571,436,207
2,++l8WoNUmsqs7C9ZVyk3pdxXklhdJcSrj50v5su2x/w=,7,2017-03-09,2017-03-28,16833.367,2404.766714,10.571429,67,12
3,++xwmVgCwpW+wr4+zw30CGbaTGYdt7yD9y+TFNI+NVs=,6,2017-03-06,2017-03-26,38017.818,6336.303000,19.500000,142,8
4,+/9G2YaP8w6Mf+xVON2swZQxSCs+PcHVOGy32DomeLk=,17,2017-03-10,2017-03-31,156576.761,9210.397706,36.647059,627,59


In [41]:
master = (
    train_sample
    .merge(members_sample, on="msno", how="left")
    .merge(transactions_agg, on="msno", how="left")
    .merge(logs_agg, on="msno", how="left")
)

print("master:", master.shape)
master.head()


master: (25000, 25)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,bd_clean,num_transactions,last_transaction_date,last_expire_date,last_plan_price,last_amount_paid,last_payment_method,last_payment_plan_days,is_auto_renew,any_cancel,active_days,first_log_date,last_log_date,total_secs_sum,total_secs_mean,num_unq_mean,num_100_sum,num_25_sum
0,jPnRUrxEFZ71o38gwQSgkyja56Mz8NUK+Q3d6yBQNYg=,0,5.0,19.0,male,7.0,2015-06-19,19.0,1.0,2017-03-12,2017-04-12,99.0,99.0,41.0,30.0,1.0,0.0,28.0,2017-03-01,2017-03-31,105572.786,3770.456643,12.428571,322.0,156.0
1,O35R1RlzfsyTF7wW6FqNfHOUiKmOfq0p4LtPLuwH+y4=,0,15.0,88.0,male,9.0,2005-11-02,88.0,1.0,2017-03-31,2017-04-30,149.0,149.0,31.0,30.0,1.0,0.0,30.0,2017-03-02,2017-03-31,364159.776,12138.659200,44.966667,1364.0,375.0
2,T2kAaQkKHy1k08W1ExIi47cUy4Q7eX00Vk5S6e46rj0=,0,3.0,21.0,male,4.0,2015-11-27,21.0,1.0,2017-03-24,2017-04-23,149.0,149.0,38.0,30.0,0.0,0.0,15.0,2017-03-01,2017-03-30,34043.494,2269.566267,12.133333,134.0,63.0
3,iGpjQPGF+N55O6WJTrEe3G8lru7L0kP2L+RaKDkj9cQ=,0,1.0,0.0,NaN,7.0,2011-10-11,NaN,1.0,2017-03-14,2017-04-14,99.0,99.0,41.0,30.0,1.0,0.0,22.0,2017-03-02,2017-03-28,58806.720,2673.032727,6.136364,234.0,17.0
4,60jtevvlQVTsNnVHqaw/UTwQ/gh1Az6LnCLSdDeNEPQ=,0,1.0,0.0,NaN,7.0,2016-02-05,NaN,1.0,2017-03-05,2017-04-05,99.0,99.0,41.0,30.0,1.0,0.0,19.0,2017-03-01,2017-03-31,60382.417,3178.021947,8.000000,217.0,43.0


In [42]:
print("Null counts after merge:")
print(master.isnull().sum())

print("\nChurn balance preserved after merge:")
print(master["is_churn"].value_counts(normalize=True))


Null counts after merge:
msno                          0
is_churn                      0
city                       2915
bd                         2915
gender                    15018
registered_via             2915
registration_init_time     2915
bd_clean                  15017
num_transactions            958
last_transaction_date       958
last_expire_date            958
last_plan_price             958
last_amount_paid            958
last_payment_method         958
last_payment_plan_days      958
is_auto_renew               958
any_cancel                  958
active_days                5638
first_log_date             5638
last_log_date              5638
total_secs_sum             5638
total_secs_mean            5638
num_unq_mean               5638
num_100_sum                5638
num_25_sum                 5638
dtype: int64

Churn balance preserved after merge:
is_churn
0    0.91004
1    0.08996
Name: proportion, dtype: float64


**Note on nulls after merge:** users with no rows in `transactions_v2.csv` or `user_logs_v2.csv`
will show as null in those columns — this itself can be a meaningful signal (e.g. inactive
users) rather than a pure data-quality problem. Decide with the team whether to impute,
flag, or drop these before feature engineering, and document that decision in the README.


In [43]:
output_path = os.path.join(OUTPUT_DIR, "master_merged.csv")
master.to_csv(output_path, index=False)
print(f"Saved merged dataset to {output_path}")
print(f"Shape: {master.shape}")


Saved merged dataset to /Users/jaswanth/KKbox/dataset/processed/master_merged.csv
Shape: (25000, 25)
